# Chapter 13 · Robust Machine Learning with ML Pipelines (guided)

**Goal for today:** train, evaluate, tune and save a model that predicts whether a recipe is a
**dessert**, using the features we prepared in Chapter 12 (cleaned in Part 1).

In Chapter 12 we fitted each preparation step by hand. Today one `Pipeline` holds every step,
so the whole thing is fitted on the training data once and reused on any new data.

| Idea | PySpark tool | Step |
|------|--------------|------|
| A **transformer** changes a data frame with `transform()` | `VectorAssembler` | 3 |
| An **estimator** learns from data with `fit()` and returns a transformer (a *model*) | `Imputer`, `MinMaxScaler`, `LogisticRegression` | 3 |
| A **pipeline** chains the stages; fitting it fits every estimator in order | `Pipeline`, `PipelineModel` | 4 to 6 |
| Models read **one vector column** of features | `VectorAssembler` | 5 |
| Learn from the training set, judge on the test set | `randomSplit` | 6 |
| Judge with more than accuracy | confusion matrix, precision, recall, `BinaryClassificationEvaluator` | 8 to 10 |
| Tune hyperparameters with **cross-validation** | `ParamGridBuilder`, `CrossValidator` | 11 |
| Explain the model | coefficients | 12 |
| Reuse the model later | `save()`, `PipelineModel.load()` | 13 |

## Step 0 · Start Spark

In [ ]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException
import pyspark.sql.functions as F
import pyspark.sql.types as T
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# change the account name to your email account
account='sli'

# define a root path to access the data in the DataAnalysisWithPythonAndPySpark
data_path='/net/clusterhn/home/'+account+'/isa460/data/'

os.environ["PYSPARK_PYTHON"] = "/opt/conda/envs/BigDataEnv/bin/python3.12"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/opt/conda/envs/BigDataEnv/bin/python3.12"

# check if the Spark session is active. If it is activate, close it

try:
    if spark:
        spark.stop()
except:
    pass    

spark = (SparkSession.builder.appName("Preparing Featurers for Machine Learning")
        .config("spark.port.maxRetries", "200")
        .config("spark.sql.mapKeyDedupPolicy", "LAST_WIN")  # This configuration allow the duplicate keys in the map data type.
        .config("spark.driver.memory", "8g")
        .getOrCreate())

# confiture the log level (defaulty is WARN)
spark.sparkContext.setLogLevel('ERROR')

---
## Step 1 · Load the cleaned recipes and look at the target
Part 1 applied the Chapter 12 cleaning and saved the result as parquet.

> **Question to consider:** About 18% of recipes are desserts. How accurate is a 'model' that always answers *not a dessert*?

In [ ]:
food = spark.read.parquet(data_path + 'recipes/recipes_cleaned')
print(food.count(), 'recipes,', len(food.columns), 'columns')
food.groupBy('dessert').count().show()

**Check.** About 20,000 recipes and 515 columns; roughly 3,600 desserts. Always answering
"not a dessert" would be about 82% accurate, so that is the bar any model has to clear,
and accuracy alone will not tell us enough.

---
## Step 2 · Name the feature groups
The model needs to know which columns are features, and each group gets different treatment.

> **Question to consider:** Why should `dessert` and `title` never be in the feature list?

In [ ]:
IDENTIFIERS = ['title']
TARGET_COLUMN = 'dessert'
CONTINUOUS_COLUMNS = ['rating', 'calories', 'protein', 'fat', 'sodium']   # imputed and scaled
RATIO_COLUMNS = ['protein_ratio', 'fat_ratio']                           # already between 0 and 1

# Everything else is a 0/1 flag (Part 1 kept only those columns)
BINARY_COLUMNS = [c for c in food.columns
                  if c not in IDENTIFIERS + [TARGET_COLUMN] + CONTINUOUS_COLUMNS + RATIO_COLUMNS]

print(len(BINARY_COLUMNS), 'binary features, for example:', BINARY_COLUMNS[:8])

**Try it.** Check that the binary columns really hold only 0 and 1. Find the largest maximum across all of them.

In [ ]:
# Your code here

In [ ]:
# Answer
food.agg(F.greatest(*[F.max(c) for c in BINARY_COLUMNS]).alias('largest max')).show()

---
## Step 3 · Transformers and estimators
Every stage of a pipeline is one or the other. An estimator has to *learn* something from the data before it can transform it.

> **Question to consider:** What does the Imputer need to learn before it can fill a missing calorie value? Does the VectorAssembler need to learn anything?

In [ ]:
import pyspark.ml.feature as MF
from pyspark.ml.base import Estimator, Transformer

imputer = MF.Imputer(
    strategy='mean',
    inputCols=CONTINUOUS_COLUMNS,
    outputCols=[c + '_i' for c in CONTINUOUS_COLUMNS],
)
continuous_assembler = MF.VectorAssembler(
    inputCols=[c + '_i' for c in CONTINUOUS_COLUMNS],
    outputCol='continuous',
)
continuous_scaler = MF.MinMaxScaler(
    inputCol='continuous',
    outputCol='continuous_scaled',
)

for stage in [imputer, continuous_assembler, continuous_scaler]:
    kind = 'estimator' if isinstance(stage, Estimator) else 'transformer'
    print(f'{type(stage).__name__:16} {kind}')

In [ ]:
# fit() turns an estimator into a model, and the model is a transformer
imputer_model = imputer.fit(food)
print(type(imputer_model).__name__, '- transformer?', isinstance(imputer_model, Transformer))
imputer_model.surrogateDF.show()      # the means it learned

---
## Step 4 · Chain the stages into a pipeline
A `Pipeline` is itself an estimator. `fit()` runs the stages in order, fitting each estimator on the output of the stages before it, and returns a `PipelineModel`.

> **Question to consider:** The scaler comes after the imputer. What would go wrong if the order were reversed?

In [ ]:
from pyspark.ml import Pipeline

food_pipeline = Pipeline(stages=[imputer, continuous_assembler, continuous_scaler])

food_pipeline_model = food_pipeline.fit(food)
food_pipeline_model.transform(food).select('continuous', 'continuous_scaled').show(3, False)

**Check.** `continuous` holds the five imputed values; `continuous_scaled` holds the same
values squeezed into 0 to 1.

---
## Step 5 · Assemble every feature into one vector
Spark models read a single vector column, conventionally called `features`.

> **Question to consider:** Most of the 513 values in each vector are 0. Why does Spark store them as a *sparse* vector?

In [ ]:
preml_assembler = MF.VectorAssembler(
    inputCols=BINARY_COLUMNS + ['continuous_scaled'] + RATIO_COLUMNS,
    outputCol='features',
)

food_pipeline.setStages([imputer, continuous_assembler, continuous_scaler, preml_assembler])

food_features = food_pipeline.fit(food).transform(food)
food_features.select('dessert', 'features').show(5, truncate=90)

**Reading a sparse vector:** `(513, [3, 40, 508], [1.0, 1.0, 0.37])` means 513 values,
all 0 except positions 3, 40 and 508, which hold 1.0, 1.0 and 0.37.

**Try it.** How many features are in the vector, and what are the names of the last three? (Hint: the names are in `food_features.schema['features'].metadata`.)

In [ ]:
# Your code here

In [ ]:
# Answer
attrs = food_features.schema['features'].metadata['ml_attr']['attrs']
names = [a['name'] for group in attrs.values() for a in sorted(group, key=lambda a: a['idx'])]
print(len(names), names[-3:])

---
## Step 6 · Add the model, split, and fit on the training data only
The classifier becomes the last stage. Fitting the pipeline on `train` means the imputer means, the scaler's min and max, and the model's weights all come from training data.

> **Question to consider:** Steps 3 to 5 fitted the stages on *all* the data. Why is that fine for a demonstration but wrong for judging a model?

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol='features', labelCol='dessert', predictionCol='prediction')

food_pipeline.setStages([imputer, continuous_assembler, continuous_scaler, preml_assembler, lr])

train, test = food.randomSplit([0.7, 0.3], 13)
train.cache()
print('training rows:', train.count(), '  test rows:', test.count())

food_pipeline_model = food_pipeline.fit(train)     # learns from train only
results = food_pipeline_model.transform(test)      # predicts the unseen test set

---
## Step 7 · Read the predictions
The model adds three columns: `rawPrediction` (the score), `probability` (not dessert, dessert) and `prediction` (0 or 1).

> **Question to consider:** Which probability cut-off turns `probability` into `prediction`?

In [ ]:
results.select('title', 'dessert', 'prediction', 'probability').show(10, truncate=40)

**Try it.** Count the test recipes the model is unsure about: probability of dessert between 0.4 and 0.6.

In [ ]:
# Your code here

In [ ]:
# Answer
from pyspark.ml.functions import vector_to_array

(results
 .withColumn('p_dessert', vector_to_array('probability')[1])
 .where(F.col('p_dessert').between(0.4, 0.6))
 .select('title', 'dessert', 'p_dessert')
 .show(truncate=50))

---
## Step 8 · The confusion matrix
Every prediction is a true or false positive or negative. Precision, recall and F1 all come from these four counts.

> **Question to consider:** For a recipe site's dessert filter, which mistake is worse: a salad shown as a dessert (false positive) or a cake that is missed (false negative)?

In [ ]:
results.groupby('dessert').pivot('prediction').count().show()

In [ ]:
cm = results.groupby('dessert').pivot('prediction').count().orderBy('dessert').collect()
TN, FP = cm[0][1], cm[0][2]
FN, TP = cm[1][1], cm[1][2]

accuracy = (TP + TN) / (TP + FP + TN + FN)
precision = TP / (TP + FP)        # of the recipes we called desserts, how many are?
recall = TP / (TP + FN)           # of the real desserts, how many did we find?
f1 = 2 * precision * recall / (precision + recall)

print(f'accuracy {accuracy:.3f}  precision {precision:.3f}  recall {recall:.3f}  f1 {f1:.3f}')

**Check.** Accuracy around 0.97, well above the 0.82 of always saying "not a dessert", with
precision and recall both above 0.9.

---
## Step 9 · Let the tools compute the metrics
The same numbers come from scikit-learn's `classification_report` (on a small pandas copy) or from the model's own `evaluate()`.

> **Question to consider:** `classification_report` shows a row for class 0 as well. Why is its recall so much higher?

In [ ]:
from sklearn.metrics import classification_report

df = results.select('dessert', 'prediction').toPandas()
print(classification_report(df['dessert'], df['prediction']))

In [ ]:
lr_model = food_pipeline_model.stages[-1]          # the fitted LogisticRegressionModel
test_summary = lr_model.evaluate(results.select('dessert', 'features'))

print('precision (dessert):', round(test_summary.precisionByLabel[1], 3))
print('recall (dessert):   ', round(test_summary.recallByLabel[1], 3))

---
## Step 10 · The ROC curve and the area under it
The ROC curve shows every trade-off between catching desserts (true positive rate) and false alarms (false positive rate) as the 0.5 cut-off moves. An area of 1.0 is perfect; 0.5 is guessing.

> **Question to consider:** Why judge a model by the area under the whole curve instead of at one cut-off?

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol='dessert', rawPredictionCol='rawPrediction', metricName='areaUnderROC')

print('Area under ROC (test):', round(evaluator.evaluate(results), 4))

In [ ]:
# The test-set ROC comes from evaluate(); lr_model.summary.roc would be the TRAINING set's
roc = test_summary.roc.toPandas()

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], 'r--')
plt.plot(roc['FPR'], roc['TPR'])
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve, test set')
plt.show()

---
## Step 11 · Tune the hyperparameters with cross-validation
`ParamGridBuilder` lists the settings to try; `CrossValidator` fits the whole pipeline once per setting per fold and keeps the setting with the best average area under ROC. Spark's default `regParam` is 0, meaning no regularization, so we try two strengths.

> **Question to consider:** This grid has 2 × 2 = 4 settings and 5 folds. How many times is the pipeline fitted?

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

grid_search = (
    ParamGridBuilder()
    .addGrid(lr.elasticNetParam, [0.0, 1.0])   # 0 = L2 (ridge), 1 = L1 (lasso)
    .addGrid(lr.regParam, [0.01, 0.1])         # regularization strength
    .build()
)

cv = CrossValidator(
    estimator=food_pipeline,
    estimatorParamMaps=grid_search,
    evaluator=evaluator,
    numFolds=5,
    seed=13,
)

cv_model = cv.fit(train)

In [ ]:
# The average area under ROC for each setting
for params, metric in zip(grid_search, cv_model.avgMetrics):
    setting = {p.name: v for p, v in params.items()}
    print(setting, round(metric, 4))

pipeline_best_model = cv_model.bestModel
print('\nTest area under ROC, best model:',
      round(evaluator.evaluate(pipeline_best_model.transform(test)), 4))

**Answer to the question:** 4 settings × 5 folds = 20 fits, plus one final fit of the best
setting on the whole training set.

**Try it.** Which regParam and elasticNetParam did the best model use? (Hint: the last stage of `pipeline_best_model` has `getRegParam()` and `getElasticNetParam()`.)

In [ ]:
# Your code here

In [ ]:
# Answer
best_lr = pipeline_best_model.stages[-1]
print('regParam:', best_lr.getRegParam(), '  elasticNetParam:', best_lr.getElasticNetParam())

---
## Step 12 · What drives the model: the coefficients
In logistic regression each feature gets a weight. Positive pushes a recipe towards *dessert*, negative away from it, and bigger means stronger (all features are on a 0 to 1 scale).

> **Question to consider:** Before running it: which features do you expect at the top, and with which sign?

In [ ]:
food_features = pipeline_best_model.transform(test)

feature_names = ['(Intercept)'] + [
    x['name'] for x in food_features.schema['features'].metadata['ml_attr']['attrs']['numeric']
]
feature_coefficients = [pipeline_best_model.stages[-1].intercept] + list(
    pipeline_best_model.stages[-1].coefficients.toArray()
)

coefficients = pd.DataFrame(feature_coefficients, index=feature_names, columns=['coef'])

coefficients['abs_coef'] = coefficients['coef'].abs()

print(coefficients.sort_values('abs_coef', ascending=False)[0:20])

**Try it.** List the ten features that push hardest towards *dessert* (the largest positive coefficients).

In [ ]:
# Your code here

In [ ]:
# Answer
coefficients.drop('(Intercept)').sort_values('coef', ascending=False).head(10)

---
## Step 13 · Save the pipeline model and use it again
The saved `PipelineModel` holds every fitted stage: the imputer means, the scaler range, the assembler's columns and the model's weights. New raw data goes in, predictions come out.

> **Question to consider:** Why save the whole pipeline instead of only the logistic regression model?

In [ ]:
from pyspark.ml.pipeline import PipelineModel

model_path = data_path + 'recipes/dessert_best_model_logisticsRegression'
pipeline_best_model.write().overwrite().save(model_path)

loaded_model = PipelineModel.load(model_path)

new_predictions = loaded_model.transform(spark.read.parquet(data_path + 'recipes/recipes_cleaned'))
new_predictions.groupBy('dessert', 'prediction').count().orderBy('dessert', 'prediction').show()

**Careful:** this scores all recipes, including the 70% the model was trained on, so it looks
better than the test-set results. Only the test set tells us how the model does on new recipes.

---
## Wrap-up

**Exit ticket** (one sentence each):
1. What is the difference between a transformer and an estimator? Give one of each.
2. Why do we fit the pipeline on `train` and not on all the data?
3. Name two metrics besides accuracy, and say what each one tells you.
4. What does cross-validation choose, and how?

In [ ]:
spark.stop()